## Задание 1

Найдите клиентов, у которых счет выглядит интересным для менеджера.

Счет можно считать таким, если он активен, город клиента известен, а баланс выше среднего баланса по всем активным счетам.

Выведите основную информацию о клиенте и счете.

Подсказка: пригодится подзапрос для расчета среднего баланса.

---

### Решение

```sql
SELECT client_name,
  city,
  account_type,
  balance,
  currency,
  opened_at
FROM bank_accounts
WHERE status = 'active' AND
  city IS NOT NULL AND
  balance > (
    SELECT AVG(balance)
    FROM bank_accounts
    WHERE status = 'active')
ORDER BY balance DESC
```

---

### Вывод результата запроса

| client_name         | city            | account_type | balance    | currency | opened_at  |
| ------------------- | --------------- | ------------ | ---------- | -------- | ---------- |
| Николай Дмитриев    | Санкт-Петербург | deposit      | 1000000.00 | RUB      | 2017-09-19 |
| Евгения Орлова      | Москва          | deposit      | 720000.00  | RUB      | 2020-04-14 |
| Екатерина Медведева | Санкт-Петербург | deposit      | 615000.00  | RUB      | 2020-09-01 |
| Денис Тихонов       | Казань          | deposit      | 510000.00  | RUB      | 2020-07-07 |
| Людмила Ефимова     | Екатеринбург    | deposit      | 430000.00  | RUB      | 2018-03-03 |
| Алексей Соколов     | Казань          | deposit      | 350000.00  | RUB      | 2019-07-10 |
| Зоя Миронова        | Сочи            | deposit      | 305000.00  | RUB      | 2021-08-28 |
| Степан Михайлов     | Москва          | deposit      | 275000.00  | RUB      | 2021-07-16 |
| Михаил Зайцев       | Сочи            | deposit      | 250000.00  | RUB      | 2023-04-01 |
| Григорий Баранов    | Екатеринбург    | debit        | 160000.00  | RUB      | 2018-06-06 |

---

## Задание 2

Покажите города, где у банка есть не меньше трех счетов.

Для каждого такого города нужно примерно оценить клиентскую базу:

- сколько всего счетов;
- какая общая сумма балансов;
- какой средний баланс;
- сколько счетов активны.

Города без названия не учитывайте.

Отсортируйте результат так, чтобы города с большим количеством счетов были выше.

---

### Решение

```sql
SELECT city,
  COUNT(*) AS number_of_accounts,
  SUM(balance) AS total_balance,
  ROUND(AVG(balance), 2) AS avg_balance,
  COUNT(CASE
          WHEN status = 'active' THEN 1
        END) AS number_of_active_accounts
FROM bank_accounts
WHERE city IS NOT NULL
GROUP BY city
HAVING COUNT(*) >= 3
ORDER BY number_of_accounts DESC
```

---

### Вывод результата запроса

| city            | number_of_accounts | total_balance | avg_balance | number_of_active_accounts |
| --------------- | ------------------ | ------------- | ----------- | ------------------------- |
| Москва          | 11                 | 1915630.75    | 174148.25   | 8                         |
| Казань          | 5                  | 997900.00     | 199580.00   | 5                         |
| Екатеринбург    | 4                  | 700661.20     | 175165.30   | 4                         |
| Санкт-Петербург | 4                  | 1678700.00    | 419675.00   | 3                         |
| Сочи            | 3                  | 675000.00     | 225000.00   | 3                         |
| Новосибирск     | 3                  | -62649.90     | -20883.30   | 2                         |
| Ростов-на-Дону  | 3                  | 47200.50      | 15733.50    | 2                         |
| Уфа             | 3                  | 150200.00     | 50066.67    | 2                         |
| Самара          | 3                  | 21500.00      | 7166.67     | 3                         |

---

## Задание 3

Найдите счета, по которым давно не было операций.

Счет стоит проверить, если последняя операция отсутствует или была раньше `2024-03-01`.

Закрытые счета можно исключить.

В результат добавьте понятный текстовый признак причины, например:

- `нет операций`;
- `операция была давно`.

Подсказка: используйте `CASE`.

---

### Решение

```sql
SELECT *,
  CASE
    WHEN last_transaction IS NULL THEN 'Нет операций'
    WHEN last_transaction < '2024-03-01' THEN 'Операция была давно'
  END AS reason
FROM bank_accounts
WHERE (last_transaction IS NULL
  OR last_transaction < '2024-03-01')
  AND status != 'closed'
```

---

### Вывод результата запроса

| account_id | client_name         | city            | account_type | balance    | currency | opened_at  | last_transaction | status  | credit_limit | manager_name  | reason              |
| ---------- | ------------------- | --------------- | ------------ | ---------- | -------- | ---------- | ---------------- | ------- | ------------ | ------------- | ------------------- |
| 3          | Алексей Соколов     | Казань          | deposit      | 350000.00  | RUB      | 2019-07-10 | null             | active  | null         | Анна Смирнова | Нет операций        |
| 8          | Наталья Федорова    | Самара          | credit       | 0.00       | RUB      | 2023-01-20 | 2024-02-10       | active  | 200000.00    | null          | Операция была давно |
| 9          | Андрей Смирнов      | null            | debit        | 1500.00    | RUB      | 2022-05-08 | 2023-12-30       | blocked | null         | Ирина Орлова  | Операция была давно |
| 13         | Михаил Зайцев       | Сочи            | deposit      | 250000.00  | RUB      | 2023-04-01 | null             | active  | null         | null          | Нет операций        |
| 14         | Анна Белова         | Санкт-Петербург | credit       | 18500.00   | RUB      | 2022-08-23 | 2024-01-21       | blocked | 80000.00     | Олег Васильев | Операция была давно |
| 16         | Евгения Орлова      | Москва          | deposit      | 720000.00  | RUB      | 2020-04-14 | null             | active  | null         | Анна Смирнова | Нет операций        |
| 22         | Ксения Григорьева   | Калининград     | debit        | 7600.00    | RUB      | 2024-02-01 | null             | active  | null         | null          | Нет операций        |
| 23         | Денис Тихонов       | Казань          | deposit      | 510000.00  | RUB      | 2020-07-07 | null             | active  | null         | Ирина Орлова  | Нет операций        |
| 26         | Алина Сергеева      | Москва          | debit        | 0.00       | RUB      | 2021-05-25 | null             | blocked | null         | Ирина Орлова  | Нет операций        |
| 27         | Николай Дмитриев    | Санкт-Петербург | deposit      | 1000000.00 | RUB      | 2017-09-19 | null             | active  | null         | Анна Смирнова | Нет операций        |
| 31         | Степан Михайлов     | Москва          | deposit      | 275000.00  | RUB      | 2021-07-16 | null             | active  | null         | Анна Смирнова | Нет операций        |
| 32         | Маргарита Соловьева | Казань          | debit        | 8900.00    | RUB      | 2023-06-21 | 2024-01-30       | active  | null         | null          | Операция была давно |
| 34         | Людмила Ефимова     | Екатеринбург    | deposit      | 430000.00  | RUB      | 2018-03-03 | null             | active  | null         | Олег Васильев | Нет операций        |
| 36         | Диана Киселева      | Москва          | credit       | 12500.00   | RUB      | 2022-04-04 | null             | active  | 85000.00     | Олег Васильев | Нет операций        |
| 38         | Екатерина Медведева | Санкт-Петербург | deposit      | 615000.00  | RUB      | 2020-09-01 | null             | active  | null         | null          | Нет операций        |
| 45         | Галина Пономарева   | Воронеж         | deposit      | 80000.00   | RUB      | 2024-02-20 | null             | active  | null         | Ирина Орлова  | Нет операций        |
| 47         | Алла Ермакова       | Ростов-на-Дону  | credit       | -8800.00   | RUB      | 2023-12-01 | null             | active  | 55000.00     | null          | Нет операций        |
| 49         | Зоя Миронова        | Сочи            | deposit      | 305000.00  | RUB      | 2021-08-28 | null             | active  | null         | Ирина Орлова  | Нет операций        |

---

## Задание 4

Оцените клиентский портфель каждого менеджера.

Для каждого менеджера нужно получить краткую сводку:

- сколько всего счетов он ведет;
- сколько среди них активных;
- какая общая сумма балансов;
- какой средний баланс;
- сколько клиентов без указанного города.

Если менеджер не указан, покажите такие счета отдельной группой, например `Без менеджера`.

Подсказка: используйте `COALESCE` и условную агрегацию.

---

### Решение

```sql
SELECT COALESCE(manager_name, 'Без менеджера') AS manager_name,
  COUNT(*) AS number_of_accounts,
  COUNT(CASE
          WHEN status = 'active' THEN 1
        END) AS number_of_active_accounts,
  SUM(balance) AS total_balance,
  ROUND(AVG(balance), 2) AS avg_balance,
  COUNT(CASE
          WHEN city IS NULL THEN 1
        END) AS number_of_accounts_without_city
FROM bank_accounts
GROUP BY COALESCE(manager_name, 'Без менеджера')
ORDER BY number_of_accounts DESC
```

---

### Вывод результата запроса

| manager_name  | number_of_accounts | number_of_active_accounts | total_balance | avg_balance | number_of_accounts_without_city |
| ------------- | ------------------ | ------------------------- | ------------- | ----------- | ------------------------------- |
| Анна Смирнова | 14                 | 10                        | 3728640.90    | 266331.49   | 1                               |
| Олег Васильев | 13                 | 10                        | 909050.80     | 69926.98    | 2                               |
| Ирина Орлова  | 13                 | 10                        | 865750.30     | 66596.18    | 2                               |
| Без менеджера | 10                 | 10                        | 925600.74     | 92560.07    | 0                               |

---

## Задание 5

Найдите кредитные счета с повышенным риском.

Счет можно считать рискованным, если:

- это кредитный счет;
- кредитный лимит указан;
- баланс отрицательный;
- использовано больше 60% кредитного лимита.

Добавьте расчет процента использования лимита.

Также можно добавить уровень риска:

- больше 90% — `высокий`;
- от 75% до 90% — `средний`;
- от 60% до 75% — `умеренный`.

Отсортируйте результат по проценту использования лимита по убыванию.

---

### Решение

```sql
SELECT account_id,
  client_name,
  city,
  balance,
  credit_limit,
  ROUND((ABS(balance) / credit_limit) * 100, 2) AS usage_limit_percent,
  CASE
    WHEN (ABS(balance) / credit_limit) * 100 > 90 THEN 'Высокий'
    WHEN (ABS(balance) / credit_limit) * 100 >= 75 THEN 'Средний'
    ELSE 'Умеренный'
  END AS risk_level
FROM bank_accounts
WHERE account_type = 'credit'
  AND credit_limit > 0
  AND balance < 0
  AND ABS(balance) > 0.6 * credit_limit
ORDER BY usage_limit_percent DESC
```

---

### Вывод результата запроса

| account_id | client_name   | city   | balance   | credit_limit | usage_limit_percent | risk_level |
| ---------- | ------------- | ------ | --------- | ------------ | ------------------- | ---------- |
| 21         | Артем Борисов | Москва | -95000.00 | 100000.00    | 95.00               | Высокий    |